# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [4]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key not set
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [5]:
# Connect to client libraries

#openai = OpenAI()

# anthropic_url = "https://api.anthropic.com/v1/"
# gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
# ollama_url = "http://localhost:11434/v1"
# openrouter_url = "https://openrouter.ai/api/v1"

    # anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
    # gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
    # grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
# ollama = OpenAI(api_key="ollama", base_url=ollama_url)
# openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [27]:
# OPENAI_MODEL = "gpt-5"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
# GEMINI_MODEL = "gemini-2.5-pro"

# Want to keep costs ultra-low? Uncomment these lines:

LLAMA_MODEL = "llama-3.3-70b-versatile"
OPENAI_MODEL = "openai/gpt-oss-120b"
OPENAI_MODEL1 = "openai/gpt-oss-20b"
QWEN_MODEL = "qwen/qwen3-32b"
GROQ_MODEL = "groq/compound"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [7]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i7-8565U CPU @ 1.80GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC Build-2) 9.2.0',
   'g++': 'g++.exe (MinGW.org GCC Build-2) 9.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [8]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = groq.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

**Short answer:**  
You already have a C++ compiler on this machine (`g++.exe` from MinGW 9.2).  
No extra installation is required to compile a single `main.cpp` file.

---

## 1. Verify the compiler is reachable

Open a Command Prompt (or PowerShell) and run:

```cmd
g++ --version
```

You should see something like:

```
g++.exe (MinGW.org GCC Build-2) 9.2.0
```

If the command is not found, add the directory that contains `g++.exe` to your `PATH`.  
Typical MinGW locations are:

* `C:\mingw64\bin`
* `C:\Program Files\mingw-w64\x86_64-9.2.0-posix-seh-rt_v6-rev0\mingw64\bin`

Add the folder to `PATH` (System Properties → Advanced → Environment Variables) and reopen the terminal.

---

## 2. Compile + run from Python

Below is a **minimal, fastest‑possible** compile‑and‑run snippet that:

* Uses the native MinGW `g++`.
* Turns on high‑level optimisations (`-O3`) and lets the compiler generate code tuned for your CPU (`-march=native`).
* Compiles as C++17 (the most recent standard comfortably supported by GCC 9).
* Produces a small executable named `main.exe` in the current working directory.

```python
import subprocess
import shlex
import sys
from pathlib import Path

# -------------------------------------------------
# 1️⃣  Compile the source
# -------------------------------------------------
compile_command = [
    "g++",                     # the compiler found in your PATH
    "-O3",                     # highest optimisation level (good for release builds)
    "-march=native",           # generate instructions for the host CPU
    "-std=c++17",              # language standard (change if you need a newer one)
    "main.cpp",                # source file (must exist in the cwd)
    "-o", "main.exe",          # output executable name
]

# Run the compilation
try:
    compile_result = subprocess.run(
        compile_command,
        check=True,            # raise CalledProcessError if compilation fails
        text=True,
        capture_output=True,   # we capture stdout/stderr for debugging
    )
    print("✅ Compilation succeeded")
    # Uncomment to see compiler warnings/errors:
    # print("STDOUT:", compile_result.stdout)
    # print("STDERR:", compile_result.stderr)
except subprocess.CalledProcessError as e:
    print("❌ Compilation failed")
    print("Return code:", e.returncode)
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    sys.exit(1)

# -------------------------------------------------
# 2️⃣  Run the executable
# -------------------------------------------------
# On Windows you can call the .exe directly; no "./" prefix is needed.
run_command = ["./main.exe"]   # works in both CMD and PowerShell when cwd is the script's dir
# (alternatively just ["main.exe"] – both are fine)

try:
    run_result = subprocess.run(
        run_command,
        check=True,
        text=True,
        capture_output=True,
    )
    print("✅ Program finished")
    print("Program output:")
    print(run_result.stdout)   # this is the value you wanted to return
except subprocess.CalledProcessError as e:
    print("❌ Program crashed")
    print("Return code:", e.returncode)
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    sys.exit(1)
```

### Why these flags?

| Flag | Reason |
|------|--------|
| `-O3` | Aggressive optimisation (inlining, vectorisation, etc.). |
| `-march=native` | Lets GCC emit instructions specific to your i7‑8565U (e.g., AVX, SSE4). |
| `-std=c++17` | Modern, stable language version supported by GCC 9. |
| `-pipe` (optional) | Uses pipes instead of temporary files for faster compilation; omitted here because the command is already short. |
| `-static` (optional) | Produces a statically‑linked binary, but on Windows this often requires additional libraries and can dramatically increase exe size, so it’s left out for simplicity. |

If you ever need **debug builds** instead, replace `-O3` with `-O0 -g`.

---

## 3. (Optional) Installing a newer compiler

If you ever want a newer GCC/Clang or the Microsoft Visual C++ toolset, you can do it via **winget**:

```cmd
winget search gcc
winget install -e --id LLVM.LLVM   # installs clang/llvm (includes clang++)

winget search visualstudio
winget install -e --id Microsoft.VisualStudio.2022.Community
# After installing, open "Developer PowerShell for VS 2022" to get cl.exe in PATH
```

But for the current task, the bundled MinGW 9.2 works perfectly.

---

## 4. Quick test

Create a tiny `main.cpp` in the same folder as your Python script:

```cpp
#include <iostream>
int main() {
    std::cout << "Hello from C++!\n";
    return 0;
}
```

Run the Python script – you should see:

```
✅ Compilation succeeded
✅ Program finished
Program output:
Hello from C++!
```

That’s it! You’re now compiling and running C++ code from Python with the fastest reasonable settings on your Windows 11 / MinGW environment. 🚀

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [ ]:
compile_command = [
    "g++",                     # the compiler found in your PATH
    "-O3",                     # highest optimisation level (good for release builds)
    "-march=native",           # generate instructions for the host CPU
    "-std=c++17",              # language standard (change if you need a newer one)
    "main.cpp",                # source file (must exist in the cwd)
    "-o", "main.exe",          # output executable name
]
run_command = ["./main"]

## And now, on with the main task

In [10]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [11]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [12]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [13]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [14]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [15]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [16]:
run_python(pi)

Result: 3.141592656089
Execution Time: 45.811956 seconds


In [33]:
port(groq, QWEN_MODEL, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [31]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [34]:
compile_and_run()

Result: 3.141592656090
Execution Time: 0.784272 seconds

Result: 3.141592656090
Execution Time: 0.888057 seconds

Result: 3.141592656090
Execution Time: 0.709456 seconds



In [35]:
45.811956/0.784272

58.41335149029929

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")